## 對齊官方最佳實踐

要對齊 LangChain 近代（v0.2～v0.3 / 向 v1 遷移）的官方最佳實踐，建議做幾個關鍵更新：

1. 把舊的 `RetrievalQA`（舊式 Chain）改成官方範例主推的「組裝式」：
   `create_stuff_documents_chain` + `create_retrieval_chain`（LCEL / Runnable 風格） ([reference.langchain.com][1])

2. `get_relevant_documents()` 這種用法在新式 Runnable 流程中，通常改成 `retriever.invoke(query)` 或直接呼叫整條 `rag_chain.invoke({"input": ...})` ([reference.langchain.com][1])

3. 依照 0.2 之後的「整合包拆分」原則：OpenAI / Chroma 都用各自的 integration package（`langchain-openai`、`langchain-chroma`） ([docs.langchain.com][2])

下面我給你一份「等價但更貼近最新官方文件」的版本：同樣做載入 → 分割 → 向量庫 → 檢索 → 生成答案，而且程式更清楚、可擴充。

## 安裝所需套件

```shell
pip install -Uq langchain langchain-community langchain-openai langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv
```

## Example Code

In [2]:
# =========================
# 0) 理解專案結構
# =========================
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent   # 因為 notebook 在 langchain/
VECTORSTORE_DIR = PROJECT_ROOT / "data" / "vectorstore" / "chroma"

VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 1) 基本設定：路徑與環境變數
# =========================
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # 從 .env 讀取 OPENAI_API_KEY

# 此 notebook 目前在 rag-playground/langchain/ 下面
PROJECT_ROOT = Path.cwd().parent
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
VECTORSTORE_DIR = PROJECT_ROOT / "data" / "vectorstore" / "chroma"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

# print("PROJECT_ROOT =", PROJECT_ROOT)
# print("DATA_RAW_DIR exists =", DATA_RAW_DIR.exists())
# print("VECTORSTORE_DIR =", VECTORSTORE_DIR)

In [4]:
# =========================
# 2) 載入文件
# =========================
from langchain_community.document_loaders import TextLoader  # PyPDFLoader 也在這個包內
# from langchain_community.document_loaders import PyPDFLoader

loader = TextLoader("../data/raw/company_policy.txt", encoding="utf-8")
documents = loader.load()

In [5]:
# =========================
# 2) 載入文件（以 txt 為例）
#    - TextLoader：一個檔案 = 一個 Document
# =========================
from langchain_community.document_loaders import TextLoader
# PyPDFLoader 也在這個包內
# from langchain_community.document_loaders import PyPDFLoader

file_path = DATA_RAW_DIR / "company_policy.txt"
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

print("Loaded documents =", len(documents))  # 如果是載入 txt 檔，整個檔案內容會被載入到一個 document 中，故只會有一個 document
print("First doc preview:\n", documents[0].page_content[:200])
print("Metadata =", documents[0].metadata)

Loaded documents = 1
First doc preview:
 公司遠端工作政策

為了提供員工更大的工作靈活性，本公司制定以下遠端工作政策：

1. 遠端工作資格
- 所有全職員工在入職滿 6 個月後，可申請遠端工作
- 部分職位可能因工作性質不適用遠端工作

2. 工作時間要求
- 遠端工作者須維持標準工作時間：上午 9 點至下午 6 點
- 必須在團隊核心會議時間（上午 10 點至下午 4 點）保持在線

3. 設備與技術要求
- 公司提供筆記型電腦和必
Metadata = {'source': '/Users/cddrm/sandbox/rag-playground/data/raw/company_policy.txt'}


In [6]:
# =========================
# 3) 切分文件成 chunks（真正進 embedding 的單位）
# =========================
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)

splits = text_splitter.split_documents(documents)

print("Splits count =", len(splits))
print(f"First split preview:\n{splits[0].page_content[:100]}")

Splits count = 1
First split preview:
公司遠端工作政策

為了提供員工更大的工作靈活性，本公司制定以下遠端工作政策：

1. 遠端工作資格
- 所有全職員工在入職滿 6 個月後，可申請遠端工作
- 部分職位可能因工作性質不適用遠端工作




In [7]:
# =========================
# 4) 建立 / 更新向量庫（Chroma）
#    - 這一步會：
#      A) 對 splits 做 embedding
#      B) 寫入（persist）到 data/vectorstore/chroma
# =========================
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings()  # 自動讀 .env 的 OPENAI_API_KEY

vector_store = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
    collection_name="company_policy",
)

# 建立「查詢介面」 Retriever，定義之後怎麼查（可重複用）
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}, # top-k = 3
)

print("Vector store ready.")

Vector store ready.


In [8]:
# =========================
# 5) 最新做法：用 LCEL 組 RAG chain（不使用 legacy chains）
#    Pipeline:
#      question -> retriever -> format_docs -> prompt -> model -> parser
# =========================
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 註：LangChain 的 OpenAI 整合會隨 OpenAI 的 Responses API 演進，
# 用 ChatOpenAI 這種方式就是目前官方整合主流入口之一。
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template(
    "你是一位企業政策助理。只能根據「文件內容」回答，不能憑空推測。"
    "如果文件沒有提到，就回答「文件未提及」。\n\n"
    "文件內容：\n{context}\n\n"
    "問題：\n{question}\n\n"
    "請用繁體中文回答。"
)

def format_docs(docs):
    """
    把「retriever 回傳的資料結構」，轉成「LLM 真正能理解、能好好用的文字上下文」。
    你也可以在這裡加上來源標記，例如 source/page
    """
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {
        "context": retriever | format_docs, # Python function 是一個 Callable, 在 LangChain 邏輯裡會被視為 Runnable (隱式轉換)
        "question": lambda q: q,
    }
    | prompt
    | llm
    | StrOutputParser()
)

# LCEL 有一個規則：
# 如果一個 dict 的 value 全部都是 Runnable（或 Callable），
# 那這個 dict 本身就會被視為一個 RunnableMap。
# RunnableMap({
#   "context": RunnableSequence(retriever, format_docs),
#   "question": RunnableLambda(lambda q: q),
# })
# LangChain在背後幫你包好了。


In [10]:
# =========================
# 6) 試跑
# =========================

# question = "公司的遠端工作政策是什麼？"
question = "公司的設備與技術要求為何？"
answer = rag_chain.invoke(question)
print(answer)

公司的設備與技術要求為：

- 公司提供筆記型電腦和必要軟體
- 員工需確保穩定的網路連線
- 須安裝公司指定的安全軟體


## 說明

我會怎麼判定「需要更新」？

第一個最大點：你用的是 `RetrievalQA.from_chain_type(...)`。這在 LangChain 現在的文件裡仍看得到參考頁，但官方在教學與主線範例上更偏向用「create_xxx_chain」把流程組起來，因為它更透明：你能清楚看到 prompt、LLM、retriever 之間怎麼接起來，也更好插入後續能力（例如：加上 query rewrite、加上 chat history、加上 rerank）。([reference.langchain.com][1])

第二個點：你原本會先 `retriever.get_relevant_documents(user_question)` 再印出文件片段。新版思路通常是：
要嘛 `retriever.invoke(query)`（把 retriever 當 runnable 叫出來），要嘛直接 `rag_chain.invoke({"input": ...})` 一次拿到 `answer` + `context`，少一段「手動搬運」的 glue code。([reference.langchain.com][1])

第三個點：套件與 import 路徑。現在 OpenAI / Chroma 這類整合，官方文件會明確寫你要裝 `langchain-openai`、`langchain-chroma`，而不是期待全部都從 `langchain` 主包拿到。這也是 v0.2 之後的「整合不可知」方向。([docs.langchain.com][2])

給你一個很具體的使用畫面：
你跑上面程式後，`result` 裡面會同時有：

* `result["answer"]`：模型的回答
* `result["context"]`：實際被檢索到、餵進 prompt 的文件片段（你印前 200 字就能驗證是不是抓對）([reference.langchain.com][1])

你下一步最值得做的一個小升級（很務實）：把 `chunk_size / chunk_overlap / k` 做成可調參數，然後用同一組測試問題（你截圖最後那幾題）跑一輪，看哪一組最穩，這就是最簡單的 RAG evaluation 起手式。

[1]: https://reference.langchain.com/v0.3/python/langchain/chains/langchain.chains.retrieval_qa.base.RetrievalQA.html?utm_source=chatgpt.com "RetrievalQA — 🦜🔗 LangChain documentation"
[2]: https://docs.langchain.com/oss/python/integrations/chat/openai?utm_source=chatgpt.com "ChatOpenAI integration - Docs by LangChain"

## 向量資料庫檔案結構


- chroma.sqlite3 是中樞（metadata / documents）
- 那串 UUID 資料夾是實際的向量索引（ANN index）
- 這整個 vectorstore/chroma/ 可以視為「一個可刪可重建的黑盒子」

```shell
# 這其實是 「關聯式資料庫 + 向量索引引擎」的組合。

vectorstore/chroma/
├── chroma.sqlite3
└── 2691687b-a610-456c-95a2-ebd2df022a33/
    ├── data_level0.bin
    ├── header.bin
    ├── length.bin
    └── link_lists.bin
```

### ① `chroma.sqlite3` 是什麼？

👉 **這是一個 SQLite 資料庫**

它負責存「非向量」的所有東西，例如：

* Document / chunk 的文字內容
* metadata（source、page、chunk id 等）
* collection 資訊
* id 對應關係（文字 ↔ 向量）

你可以把它想成：

> **Chroma 的「控制平面 + 目錄」**

如果你用 SQLite viewer 打開，會看到很多 table（不用手動改）。

📌 **重點**

* 它不存高維向量
* 它負責「語意資料、結構、對應關係」

### ② 那串 UUID 資料夾是什麼？

```text
2691687b-a610-456c-95a2-ebd2df022a33/
```

👉 **這是一個 Chroma collection 的實體目錄**

* 每一個 collection → 一個 UUID
* 你沒手動命名時，Chroma 自動生成
* 你這次大概只有一個 collection，所以只有一個 UUID

### ③ 裡面的 `.bin` 檔案在幹嘛？

這一組 `.bin` 檔案是 **向量搜尋引擎的核心索引**（ANN index），通常是基於 **HNSW（Hierarchical Navigable Small World）**。

你不用記名字，但知道用途即可。

- `data_level0.bin`

  * 存放實際的向量資料（embedding）
  * 高維 float array
  * **最大、最重要**

- `link_lists.bin`

  * HNSW 的「圖結構」
  * 決定向量之間怎麼走捷徑找最近鄰

- `header.bin`

  * 索引的設定資訊
  * 維度、參數、版本等

- `length.bin`

  * 記錄向量數量、offset 等輔助資訊

👉 **這四個檔案合起來 =「向量搜尋引擎本體」**

### ④ 為什麼要「SQLite + bin 檔」分開？

這其實是設計優點，不是怪異行為：

* SQLite：

  * 擅長結構化資料、metadata、交易
* Bin 檔（ANN index）：

  * 擅長高速數值運算與近似搜尋

👉 **這是所有成熟向量庫的共通模式**
（FAISS / Milvus / Qdrant 都類似，只是形式不同）

### ⑤ 這個資料夾你「能不能動？」

- ❌ 不要手動改、刪單一檔案 -> 會直接壞 index。
- ✅ 可以整個刪掉，然後重新跑：

```python
Chroma.from_documents(...)
```

👉 **這就是為什麼我們把它放在 `data/vectorstore/`，並加進 `.gitignore`**

### ⑥ 對你現在專案來說，該怎麼看待它？

請用這個心智模型：

> **`vectorstore/chroma/` = build artifact，不是 source code**

* 不需要 commit
* 不需要理解內部格式
* 不需要穩定 API
* 只需要知道：
  **「它來自 data/raw + splitter + embeddings」**

---

### 最後一句話，幫你完全去魅

> **看到這個資料夾，代表你的 RAG「索引期」已經成功完成。**
> 它的存在，本身就是成功的證據，而不是需要你維護的東西。


## 說明 Runnable Pipeline

> **在 LCEL 裡，這個 dict 不是普通的 Python dict，
> 而是被 LangChain 視為一個「RunnableMap」。**

也就是說：

```python
{
    "context": retriever | format_docs,
    "question": lambda q: q,
}
```

在 LCEL 的語意裡是：

> **「把同一個輸入，同時送進多條 Runnable，
> 然後把結果組成一個 dict 輸出。」**

從**純 Python**角度看：

* dict 只是資料
* 資料不能被 `| prompt`

所以你會自然問：

> 「這不是一個 dict 嗎？怎麼能接 Runnable？」

答案是：
👉 **LCEL 偷偷把這個 dict「升級」成一個 Runnable**

### LangChain 在這裡做了什麼事？（重點）

LCEL 有一個規則：

> **如果一個 dict 的 value 全部都是 Runnable（或 Callable），
> 那這個 dict 本身就會被視為一個 RunnableMap。**

所以你這段其實等價於（概念上）：

```text
RunnableMap({
  "context": RunnableSequence(retriever, format_docs),
  "question": RunnableLambda(lambda q: q),
})
```

你沒有寫，但 LangChain在背後幫你包好了。

### 那這個 RunnableMap 在幹嘛？

假設你呼叫：

```python
rag_chain.invoke("公司的遠端工作政策是什麼？")
```

RunnableMap 的行為是：

1. 接收輸入 `q = "公司的遠端工作政策是什麼？"`
2. **同時**做兩件事（平行概念，不是 async）：

   * 把 `q` 丟進 `retriever | format_docs`
   * 把 `q` 丟進 `lambda q: q`
3. 組成一個 dict：

```python
{
    "context": "（檢索後拼好的文件文字）",
    "question": "公司的遠端工作政策是什麼？"
}
```

4. 把這個 dict 當成**下一個 Runnable 的輸入**

### 為什麼這個設計很重要？（不是語法糖）

因為這讓你可以：

* **同一份輸入**
* **分流成多個語意角色**
* 再在 prompt 層重新匯合

你現在的 pipeline 本質是：

```text
問題字串
   ↓
┌───────────────┐
│ RunnableMap   │
│               │
│ q → context   │
│ q → question  │
└───────────────┘
        ↓
     prompt
```

這不是為了炫技，而是為了**清楚地表達資料流的語意**。

### 為什麼 prompt 可以接在後面？

因為：

* `RunnableMap` 的輸出是一個 dict
* `ChatPromptTemplate` 的輸入**正是 dict**
* key 對應 template 裡的 `{context}`、`{question}`

所以：

```python
RunnableMap | prompt
```

在型別上是完全對齊的。

這其實等價於你手寫的老派程式碼

如果不用 LCEL，你可能會寫成這樣：

```python
def rag(q):
    context = format_docs(retriever.invoke(q))
    question = q
    messages = prompt.format_messages(
        context=context,
        question=question,
    )
    response = llm.invoke(messages)
    return response.content
```

LCEL 只是把這件事「**變成一條可組合、可插拔、可觀測的管線**」。

### 一句話幫你完全收掉

> **在 LCEL 裡，只要 dict 的 value 是 Runnable / Callable，
> 這個 dict 就不再是資料，而是「一個並行轉換節點（RunnableMap）」。**

